# Prove the cumulative handoff

Use the selected alpha, validated stage and active batch path together. The reference alpha=0.01 is selected from the fixed validation matrix; it is not a universal hyperparameter recommendation.

**Local:** [set up once](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/docs/setup.md), select the module `.venv` kernel, then run cells in order. **Colab:** CPU, runtime version **2026.07** (Python 3.12). If setup asks for a restart after installing dependencies, use **Restart session**, then run again. No API key is needed. Setup prints a new temporary output folder; save your work before ending the session.

Use this one notebook throughout the sprint: predict → run the guided cells → add your answers in **Your turn** → finish the checkpoint. Campus notes explain the same work; you do not repeat it in a second notebook.

<details><summary>Campus notes for this sprint</summary>

- [Scoping the cumulative pipeline for production-grade reproducibility](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-4/01-scoping-the-cumulative-pipeline-for-production-grade-reproducibility/lesson.md)
- [Integrating pipeline, tracking, and registry](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-4/02-integrating-pipeline-tracking-and-registry/lesson.md)
- [Proving the selected serving path uses the active version](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-4/03-proving-the-selected-serving-path-uses-the-active-version/lesson.md)
- [Setup, run, and expected-evidence instructions](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-4/04-setup-run-and-expected-evidence-instructions/lesson.md)
- [Proving the pipeline reproduces](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-4/05-proving-the-pipeline-reproduces/lesson.md)
- [Reproducible pipeline deliverable](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-4/06-reproducible-pipeline-deliverable/lesson.md)

</details>

In [1]:
from pathlib import Path
import json, tempfile, sys, subprocess
PIN = {'package': 'campus-mlops', 'version': '0.1.1', 'code_revision': 'sha256:a7a3061850f1f1eaa158754e719262b26a000af73019ada402be743c6f66e6c2', 'repository': 'https://github.com/richhiey/ml-llm-ops.git', 'source_commit': 'f1ae85914f49a4f1c4bdd6a0ca62c4c4076158ea', 'package_path': 'module-a'}
assert sys.version_info[:2] == (3, 12), 'Use Python 3.12; see the linked setup guide.'
if 'google.colab' in sys.modules:
    # Start before importing numerical packages: installed files cannot replace live modules.
    expected = {'numpy':'2.3.3', 'pandas':'2.3.3', 'sklearn':'1.7.2', 'mlflow':'3.5.1'}
    stale = [name for name, version in expected.items()
             if name in sys.modules and getattr(sys.modules[name], '__version__', None) != version]
    checkout = Path('/content') / ('ml-llm-ops-' + PIN['source_commit'][:12])
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--no-checkout', PIN['repository'], str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', PIN['source_commit']], check=True)
    actual = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'], text=True).strip()
    assert actual == PIN['source_commit']
    package = checkout / PIN['package_path']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(package/'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest==8.4.2'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(package)], check=True)
    if stale:
        raise RuntimeError('Pinned dependencies are now installed. Choose Runtime > Restart session, '
                           'then Run all again. Keep this runtime: do NOT disconnect/delete it. '
                           'Loaded packages needing refresh: ' + ', '.join(stale))
    # A running kernel does not reload newly installed editable .pth files.
    sys.path.insert(0, str(package / 'src'))
import campus_mlops
from campus_mlops import *
from campus_mlops.lifecycle import code_revision, MODEL_NAME
PACKAGE=Path(campus_mlops.__file__).resolve().parents[2]
sys.path.insert(0,str(PACKAGE))
from learning.workspace import make_project, project_python, project_tests, course_cli, prepare_case
assert campus_mlops.__version__ == PIN['version'] == '0.1.1'
assert code_revision() == PIN['code_revision'], 'Helper changed: restore the pinned reference source.'
state = Path(tempfile.mkdtemp(prefix='campus-module-a-'))
print('Helper', campus_mlops.__version__, 'source commit:', PIN['source_commit'], 'state:', state)


Helper 0.1.1 source commit: f1ae85914f49a4f1c4bdd6a0ca62c4c4076158ea state: /var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-rcs9rb6u


## Integrate an end-to-end run
Build from a new state with guard, explicit metrics, autologging and registry evidence. No orchestration framework is needed for this sequence.

In [2]:
ws=Workspace(state/'handoff')
record=ws.run(Config(alpha=0.01,range_guard=True,autolog=True),name='handoff')
version=register(ws,record['run_id'])
approve(ws,version,'Controlled alpha comparison and range guard checks')
print(activate(ws,version,'Final validated selection'))
output,inference=batch_predict(ws)
print('Run:',record['run_id'],'validation:',record['metrics']); print(inference)
assert inference['rows']==24 and inference['complete_within_60s']

2026/10/19 17:38:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/19 17:38:22 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 17:38:22 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 17:38:22 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-10-19 17:38:22 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


...

2026/10/19 17:38:22 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 17:38:22 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 17:38:22 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


{'timestamp_utc': '2026-10-19T15:38:23.023195+00:00', 'model': 'dev.linden-demand', 'alias': 'active', 'previous_version': None, 'new_version': '1', 'run_id': '865dd2388fcd4417a1a2ef8eca9e9b56', 'reason': 'Final validated selection'}
Run: 865dd2388fcd4417a1a2ef8eca9e9b56 validation: {'rmse_kwh': 1.4342817684933908, 'mae_kwh': 1.201891273831005}
{'resolved_version': '1', 'run_id': '865dd2388fcd4417a1a2ef8eca9e9b56', 'model': 'dev.linden-demand', 'rows': 24, 'elapsed_seconds': 0.0076608750387094915, 'rows_per_second': 3132.8013939309612, 'complete_within_60s': True, 'input_sha256': '5b74deb237ea9a085666ab0b3f865db8c7c996b935b38b24e829567d89142ee6'}


Successfully registered model 'dev.linden-demand'.
Created version '1' of model 'dev.linden-demand'.


## Inspect one connected identity chain

Predict which identities should agree before reading them. The registered artifact must point to the selected training run, and the new batch must report that source run and version.

In [3]:
selected=ws.client.get_model_version_by_alias(MODEL_NAME,'active')
assert selected.run_id==record['run_id']==inference['run_id']
assert str(selected.version)==str(version)==inference['resolved_version']
print('Run → version → batch:',record['run_id'],version,inference['resolved_version'])

Run → version → batch: 865dd2388fcd4417a1a2ef8eca9e9b56 1 1


## Keep a failed candidate from changing active

This is an intentionally rejected candidate, not another configuration to promote. Record active before/after; registration can succeed even when approval fails.

In [4]:
before=str(ws.client.get_model_version_by_alias(MODEL_NAME,'active').version)
rejected=ws.run(Config(alpha=100,range_guard=True),name='rejected-candidate')
rejected_version=register(ws,rejected['run_id'])
try:
    approve(ws,rejected_version,'Deliberate failing gate probe')
except ValueError as error:
    print('Expected rejection:',error)
else:
    raise AssertionError('Invalid approval accepted')
after=str(ws.client.get_model_version_by_alias(MODEL_NAME,'active').version)
assert before==after

/Users/richhiey/Desktop/projects/masterschool/lessons/mlops-llmops/shared/ml-llm-ops/module-a/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/19 17:38:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a v

Expected rejection: Approval needs rationale, range_guard=True and finite validation RMSE <= 5 kWh


Registered model 'dev.linden-demand' already exists. Creating a new version of this model...
Created version '2' of model 'dev.linden-demand'.


## Reproduce before handoff
The recorded source/data/environment must match. Replaying to a new store tests absence of hidden registry and notebook dependencies.

In [5]:
report=reproduce(ws,record['run_id'],state/'fresh-reproduction')
print(report)
assert report['passed'] and report['lineage_matched']

2026/10/19 17:38:23 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/19 17:38:23 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 17:38:23 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 17:38:23 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-10-19 17:38:23 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


...

2026-10-19 17:38:23 INFO  [alembic.runtime.migration] Running upgrade 3da73c924c2f -> bf29a5ff90ea, add jobs table


2026-10-19 17:38:23 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 17:38:23 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026/10/19 17:38:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{'original_run_id': '865dd2388fcd4417a1a2ef8eca9e9b56', 'reproduced_run_id': 'f0894093a4ff48d2b5eaa3dd8a391113', 'rmse_delta': 0.0, 'tolerance': 1e-06, 'passed': True, 'lineage_matched': True}


## Evaluate the reserved final holdout once
After selection, measure final rows without tuning against their result. Keep this run out of the validation comparison group.

Configuration selection is already complete in this reference. Report this result separately; do not retune alpha against it or insert it into the validation comparison. The approval policy is validation-based, so final-partition observations cannot be promoted through that gate.

In [6]:
final=ws.run(Config(alpha=0.01,range_guard=True,autolog=True),name='final-holdout',partition='final')
print(final['metrics'])
assert final['evaluation_partition']=='final'

2026/10/19 17:38:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{'rmse_kwh': 1.2148472211584365, 'mae_kwh': 0.9971419477925263}


## Write a verifiable handoff index
The complete submission also includes your explained extension diff and setup instructions. Check each named file exists; an index is a locator, not proof by itself.

In [7]:
handoff={'version':version,'run_id':record['run_id'],'code_revision':record['code_revision'],
'data_sha256':record['data_sha256'],'config':record['config'],'environment':record['environment'],
'model_uri':record['model_uri'],'inference':inference,'reproduction':report,
'exact_training_command':record['command'],'tolerance_rmse_kwh':1e-6}
(state/'handoff.json').write_text(json.dumps(handoff,indent=2))
print('Saved handoff index:',state/'handoff.json'); print('Selected version:',version,'reproduction passed:',report['passed'])
assert (ws.path/'batch.csv').exists() and (ws.path/'audit.jsonl').exists()

Saved handoff index: /var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-rcs9rb6u/handoff.json
Selected version: 1 reproduction passed: True


## Inspect what the receiver must recover

The notebook replay above used the same environment. A stronger handoff restores the source archive to a new folder, creates a new locked environment and uses a new state. The following inspects real recovery content; follow the full Campus handoff activity for the independent receiving run.

In [8]:
source=Path(ws.client.download_artifacts(record['run_id'],'reproduction/source.tar.gz'))
print('Recoverable source bytes:',source.stat().st_size)
print('Recorded command:',record['command'])
print('Tolerance:',report['tolerance'],'delta:',report['rmse_delta'])
assert source.is_file() and report['lineage_matched']

Recoverable source bytes: 79439
Recorded command: campus-mlops train --state fresh-state --alpha 0.01 --range-guard --autolog --partition validation
Tolerance: 1e-06 delta: 0.0


## Your turn: hand over one working project

Continue with your actual edited project and selected configuration. Make a short inventory: what works, what is missing, and which missing item blocks the receiver. Adapt the demonstrated chain to that project using `project_python`/`course_cli` (fresh processes), rather than importing edited source over the reference kernel. Retain a failed approval with active unchanged. Check batch row order, finite outputs, resolved version and whole-job time; run invalid input separately. Write the exact setup/run instructions alongside the source. Follow [fresh-environment recovery](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/docs/clean-state.md) locally: recover source to a new folder, create its locked environment and use a new store. Compare lineage and RMSE within 1e-6 kWh. Use one project and one handoff index for all these activities.

## Sprint checkpoint

Hand Harbor your working project, setup/run instructions and compact handoff index. A receiver must recover source/data/configuration/environment, build a new locked environment, rerun in a new store, reproduce within 1e-6 kWh and produce the required 24-row batch within 60 seconds. Apply the Campus final acceptance brief, retaining any failure and its fix. The reference replay above shares a kernel and does not replace this independent receiving run. An optional T4 model transfer stays within the same contracts and project.

## Finish and keep your work

Save this notebook and the linked project/output files. Keep one sprint submission, reusing the work already completed. Temporary tracking directories disappear when the environment is discarded; save the files before closing it. Continue with the same project in the next sprint.